# Aging Narratives 2024 - Data Cleaning Notebook

This notebook applies the cleaning techniques from Otto's `cleaned_longevity_data.py` to the 2024 articles dataset, with **enhanced text cleaning** to remove web scraping noise.

## Cleaning Steps:
1. Load the raw data and inspect
1b. **Clean article text content** (remove social media, navigation, boilerplate noise)
2. Remove NULL URLs and duplicates
3. Text validation (minimum characters, words, punctuation)
4. Standardize province names
5. Clean text fields (title, phrase, article_text)
6. Clean and validate phrase_category
7. Date standardization (extract year, month)
8. Add metadata (ingestion_date, row_hash)
9. Save cleaned data

---
## Step 0: Import Libraries

In [29]:
import pandas as pd
import numpy as np
from datetime import datetime
import hashlib
import re

print("Libraries imported successfully!")

Libraries imported successfully!


---
## Step 1: Load the Raw Data

Load the 2024 articles dataset and inspect its structure.

In [152]:
# Load the 2024 dataset
'''
INPUT_FILE = '../newData/aging_narratives_articles_2024.csv'
OUTPUT_FILE = '../newData/cleanData/aging_narratives_articles_2024_cleaned.csv'

INPUT_FILE = '../newData/agingnarrativesarticles2023.csv'
OUTPUT_FILE = '../newData/cleanData/agingnarrativesarticles2023_cleaned.csv'

'''
INPUT_FILE = '../newData/agingnarrativesarticles20252026.csv'
OUTPUT_FILE = '../newData/cleanData/aagingnarrativesarticles20252026_cleaned.csv'

df = pd.read_csv(INPUT_FILE)

initial_count = len(df)
print(f"Loaded {initial_count} total rows from {INPUT_FILE}")
print(f"\nColumns: {list(df.columns)}")

Loaded 16149 total rows from ../newData/agingnarrativesarticles20252026.csv

Columns: ['url', 'title', 'publish_date', 'media_name', 'phrase', 'phrase_category', 'province', 'article_text']


In [153]:
# Display first few rows
df.head()

,url,title,publish_date,media_name,phrase,phrase_category,province,article_text
0,https://www.therecord.com/news/canada/close-2-...,Close 2 Home Caledon obtains charitable status...,2026-01-29,therecord.com,aging crisis,limiting,Ontario,After running their non-profit organization fo...
1,https://www.wellandtribune.ca/news/canada/clos...,Close 2 Home Caledon obtains charitable status...,2026-01-29,wellandtribune.ca,aging crisis,limiting,Ontario,After running their non-profit organization fo...
2,https://www.niagarafallsreview.ca/news/canada/...,Close 2 Home Caledon obtains charitable status...,2026-01-29,niagarafallsreview.ca,aging crisis,limiting,Ontario,After running their non-profit organization fo...
3,https://www.thespec.com/news/canada/close-2-ho...,Close 2 Home Caledon obtains charitable status...,2026-01-29,thespec.com,aging crisis,limiting,Ontario,After running their non-profit organization fo...
4,https://www.theglobeandmail.com/opinion/articl...,"In an uncertain climate, Canada needs a civil ...",2026-01-29,theglobeandmail.com,aging crisis,limiting,Ontario,Marcus Kolga is a senior fellow at the Macdona...


In [154]:
# Check data types and null counts
print("Data Types and Null Counts:")
print("-" * 50)
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
print(info_df)

Data Types and Null Counts:
--------------------------------------------------
                dtype  null_count  null_pct
url               str           0      0.00
title             str           0      0.00
publish_date      str           0      0.00
media_name        str           0      0.00
phrase            str           0      0.00
phrase_category   str           0      0.00
province          str           0      0.00
article_text      str        1525      9.44


---
## Step 1b: Clean Article Text Content (Remove Noise)

The scraped articles often contain noise like:
- Social media buttons (Facebook, Twitter, WhatsApp, SMS, Email, etc.)
- Navigation elements (Tags, Most Popular, Latest News, etc.)
- Newsletter signup text (Subscribe, Sign up, Newsletter, etc.)
- Boilerplate headers/footers (The Canadian Press, Submit Your News, etc.)
- Excessive whitespace and blank lines

This step removes these common noise patterns from the article text before validation.

In [155]:
def clean_article_text(text):
    """
    Remove common noise patterns from scraped article text.
    
    Removes:
    - Social media buttons and sharing links
    - Navigation elements
    - Newsletter/subscription text
    - Boilerplate headers/footers
    - Excessive whitespace
    """
    if pd.isna(text) or text == "":
        return ""
    
    text = str(text)
    
    # ===========================================
    # 1. SOCIAL MEDIA BUTTONS & SHARING
    # ===========================================
    social_patterns = [
        r'(?<![a-zA-Z])Facebook(?![a-zA-Z])',
        r'(?<![a-zA-Z])Twitter(?![a-zA-Z])',
        r'(?<![a-zA-Z])WhatsApp(?![a-zA-Z])',
        r'(?<![a-zA-Z])SMS(?![a-zA-Z])',
        r'(?<![a-zA-Z])LinkedIn(?![a-zA-Z])',
        r'(?<![a-zA-Z])Instagram(?![a-zA-Z])',
        r'(?<![a-zA-Z])Pinterest(?![a-zA-Z])',
        r'(?<![a-zA-Z])Reddit(?![a-zA-Z])',
        r'(?<![a-zA-Z])Tumblr(?![a-zA-Z])',
        r'(?<![a-zA-Z])YouTube(?![a-zA-Z])',
        r'(?<![a-zA-Z])TikTok(?![a-zA-Z])',
        r'\bShare this\b',
        r'\bShare on\b',
        r'\bFollow us\b',
        r'\bLike us\b',
        r'\bPin it\b',
    ]
    
    for pattern in social_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    # ===========================================
    # 2. NAVIGATION & UI ELEMENTS
    # ===========================================
    nav_patterns = [
        r'\bMost Popular\b',
        r'\bLatest News\b',
        r'\bRelated Articles\b',
        r'\bRelated Stories\b',
        r'\bRead More\b',
        r'\bRead Also\b',
        r'\bSee Also\b',
        r'\bMore Stories\b',
        r'\bTop Stories\b',
        r'\bLeave a comment\b',
        r'\bPost a comment\b',
        r'\bCopy article link\b',
        r'\bGo to form\b',
    ]
    
    for pattern in nav_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    # ===========================================
    # 3. NEWSLETTER & SUBSCRIPTION TEXT
    # ===========================================
    newsletter_patterns = [
        r'Subscribe to our newsletter.*?(?=\.|$)',
        r'Sign up for our newsletter.*?(?=\.|$)',
        r'Receive daily headlines.*?(?=\.|$)',
        r'Sign up now!',
        r'Manage your lists',
        r'Success! An email has been sent.*?(?=\.|$)',
        r'Error! There was an error.*?(?=\.|$)',
        r'\bCookie Policy\b',
        r'\bPrivacy Policy\b',
        r'\bTerms of Service\b',
        r'\bTerms and Conditions\b',
    ]
    
    for pattern in newsletter_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    # ===========================================
    # 4. BOILERPLATE HEADERS/FOOTERS
    # ===========================================
    boilerplate_patterns = [
        r'The Canadian Press\.?\s*All rights reserved\.?',
        r'Associated Press\.?\s*All rights reserved\.?',
        r'Reuters\.?\s*All rights reserved\.?',
        r'©\s*\d{4}.*?(?=\.|$)',
        r'Copyright\s*©?\s*\d{4}.*?(?=\.|$)',
        r'All rights reserved\.?',
        r'Submit Your News',
        r"We're always interested in hearing about news.*?(?=\.|$)",
        r'Let us know what\'s going on!',
        r'Submit a Letter to the Editor',
        r"If you're interested in submitting.*?(?=\.|$)",
        r'Sorry, there are no recent results.*?(?=\.|$)',
        r'\bADVERTISEMENT\b',
        r'\bSponsored Content\b',
        r'\bSponsored\b',
    ]
    
    for pattern in boilerplate_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    # ===========================================
    # 5. CLEAN UP WHITESPACE
    # ===========================================
    # Remove multiple spaces
    text = re.sub(r' +', ' ', text)
    # Remove multiple newlines (more than 2)
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Remove lines that are just whitespace
    text = re.sub(r'\n\s*\n', '\n\n', text)
    # Strip leading/trailing whitespace
    text = text.strip()
    
    return text

print("clean_article_text() function defined!")

clean_article_text() function defined!


In [156]:
# Apply the cleaning function to all articles
print("Cleaning article text...")
print("-" * 60)

# Store original lengths for comparison
original_lengths = df['article_text'].fillna('').astype(str).str.len()

# Apply cleaning
df['article_text'] = df['article_text'].apply(clean_article_text)

# Calculate new lengths
cleaned_lengths = df['article_text'].str.len()

# Statistics
total_chars_removed = (original_lengths - cleaned_lengths).sum()
avg_chars_removed = (original_lengths - cleaned_lengths).mean()

print(f"Total characters removed: {total_chars_removed:,}")
print(f"Average characters removed per article: {avg_chars_removed:.0f}")
print(f"\nBefore cleaning - Average length: {original_lengths.mean():.0f} chars")
print(f"After cleaning  - Average length: {cleaned_lengths.mean():.0f} chars")
print("-" * 60)
print("Article text cleaned!")

Cleaning article text...
------------------------------------------------------------
Total characters removed: 1,353,054
Average characters removed per article: 84

Before cleaning - Average length: 3834 chars
After cleaning  - Average length: 3750 chars
------------------------------------------------------------
Article text cleaned!


In [157]:
# Show a before/after example
# Reload original to show comparison
df_original = pd.read_csv(INPUT_FILE)

# Find an article with significant noise removed
sample_idx = (original_lengths - cleaned_lengths).idxmax()

print("=" * 60)
print("BEFORE/AFTER COMPARISON (Sample Article)")
print("=" * 60)
print(f"\nArticle Index: {sample_idx}")
print(f"Original length: {original_lengths[sample_idx]} chars")
print(f"Cleaned length: {cleaned_lengths[sample_idx]} chars")
print(f"Characters removed: {original_lengths[sample_idx] - cleaned_lengths[sample_idx]}")

print("\n" + "-" * 60)
print("BEFORE (first 500 chars):")
print("-" * 60)
print(str(df_original.loc[sample_idx, 'article_text'])[:500])

print("\n" + "-" * 60)
print("AFTER (first 500 chars):")
print("-" * 60)
print(df.loc[sample_idx, 'article_text'][:500])

BEFORE/AFTER COMPARISON (Sample Article)

Article Index: 12178
Original length: 4558 chars
Cleaned length: 2999 chars
Characters removed: 1559

------------------------------------------------------------
BEFORE (first 500 chars):
------------------------------------------------------------
First human case of West Nile virus acquired in Canada this year confirmed


1st human case of West Nile acquired in Canada this year confirmed



Nicole Ireland The Canadian Press


            Jul 31, 2025
        

Jul 31, 2025













Facebook






Twitter






WhatsApp






SMS






Email



















File - In this Aug. 26, 2019, file photo, Salt Lake City Mosquito Abatement District biologist Nadja Reissen examines a mosquito. (AP Photo/Rick Bowmer, File)


RB













------------------------------------------------------------
AFTER (first 500 chars):
------------------------------------------------------------
First human case of West Nile virus acquired in Canada this y

---
## Step 2: Remove NULL URLs and Duplicates

Following Otto's approach:
- Remove rows where URL is NULL
- Remove duplicate URLs (keep first occurrence based on publish_date)

In [158]:
# Remove NULL URLs
df_clean = df[df['url'].notna()].copy()
after_null_filter = len(df_clean)

print(f"After removing NULL URLs: {after_null_filter} rows")
print(f"Removed: {initial_count - after_null_filter} rows with NULL URLs")

After removing NULL URLs: 16149 rows
Removed: 0 rows with NULL URLs


In [159]:
# Remove duplicates by URL (keep first occurrence based on publish_date)
# Sort by publish_date first to ensure we keep the earliest
df_clean = df_clean.sort_values('publish_date')
df_clean = df_clean.drop_duplicates(subset=['url'], keep='first')

after_dedup = len(df_clean)

print(f"After deduplication: {after_dedup} rows")
print(f"Removed: {after_null_filter - after_dedup} duplicate URLs")

After deduplication: 15317 rows
Removed: 832 duplicate URLs


In [160]:
df_clean.shape

(15317, 8)

---
## Step 3: Text Validation

Create validation flags for article text quality:
- **has_min_chars**: Article text has at least 200 characters
- **has_min_words**: Article text has at least 30 words
- **has_punctuation**: Article text contains periods or question marks
- **is_not_empty**: Article text is not null and not blank

Only articles passing ALL checks are kept.

In [161]:
# Clean article text first (strip whitespace)
df_clean['article_text_clean'] = df_clean['article_text'].fillna('').astype(str).str.strip()

# Validation checks
df_clean['has_min_chars'] = df_clean['article_text_clean'].str.len() >= 200

# Count words (split by whitespace)
df_clean['word_count'] = df_clean['article_text_clean'].str.split().str.len().fillna(0)
df_clean['has_min_words'] = df_clean['word_count'] >= 30

# Check for punctuation (. or ?)
df_clean['has_punctuation'] = (
    df_clean['article_text_clean'].str.contains(r'[.?]', regex=True, na=False)
)

# Check not empty
df_clean['is_not_empty'] = (
    df_clean['article_text'].notna() & 
    (df_clean['article_text_clean'] != '')
)

# Overall text_valid flag
df_clean['text_valid'] = (
    df_clean['has_min_chars'] & 
    df_clean['has_min_words'] & 
    df_clean['has_punctuation'] & 
    df_clean['is_not_empty']
)

print("Validation flags created!")
print("\nValidation Summary:")
print("-" * 50)
print(f"has_min_chars (>=200 chars):  {df_clean['has_min_chars'].sum()} passed, {(~df_clean['has_min_chars']).sum()} failed")
print(f"has_min_words (>=30 words):   {df_clean['has_min_words'].sum()} passed, {(~df_clean['has_min_words']).sum()} failed")
print(f"has_punctuation (. or ?):     {df_clean['has_punctuation'].sum()} passed, {(~df_clean['has_punctuation']).sum()} failed")
print(f"is_not_empty:                 {df_clean['is_not_empty'].sum()} passed, {(~df_clean['is_not_empty']).sum()} failed")
print("-" * 50)
print(f"text_valid (all checks):      {df_clean['text_valid'].sum()} passed, {(~df_clean['text_valid']).sum()} failed")

Validation flags created!

Validation Summary:
--------------------------------------------------
has_min_chars (>=200 chars):  13959 passed, 1358 failed
has_min_words (>=30 words):   13985 passed, 1332 failed
has_punctuation (. or ?):     14073 passed, 1244 failed
is_not_empty:                 14128 passed, 1189 failed
--------------------------------------------------
text_valid (all checks):      13938 passed, 1379 failed


In [162]:
# Filter to only valid articles
invalid_count = (~df_clean['text_valid']).sum()
df_clean = df_clean[df_clean['text_valid']].copy()

# Drop intermediate validation columns
df_clean = df_clean.drop(columns=['has_min_chars', 'has_min_words', 'has_punctuation', 
                                   'is_not_empty', 'word_count', 'text_valid'])

after_validation = len(df_clean)
print(f"After text validation: {after_validation} rows")
print(f"Removed: {invalid_count} rows with invalid article text")

After text validation: 13938 rows
Removed: 1379 rows with invalid article text


---
## Step 4: Standardize Province Names

Clean and standardize province names to:
- Ontario
- British Columbia
- Quebec
- Alberta
- Other (for any unrecognized)

In [163]:
# Check current province values
print("Current province values:")
print(df_clean['province'].value_counts())

Current province values:
province
Ontario             7041
British Columbia    4361
Alberta             2386
Quebec               150
Name: count, dtype: int64


In [164]:
def standardize_province(province):
    """Standardize province names following Otto's logic"""
    if pd.isna(province):
        return 'Other'
    
    province_lower = str(province).lower()
    
    if 'ontario' in province_lower:
        return 'Ontario'
    elif 'british columbia' in province_lower or province_lower == 'bc':
        return 'British Columbia'
    elif 'quebec' in province_lower or 'québec' in province_lower:
        return 'Quebec'
    elif 'alberta' in province_lower:
        return 'Alberta'
    else:
        return 'Other'

df_clean['province_clean'] = df_clean['province'].apply(standardize_province)

print("\nStandardized province values:")
print(df_clean['province_clean'].value_counts())


Standardized province values:
province_clean
Ontario             7041
British Columbia    4361
Alberta             2386
Quebec               150
Name: count, dtype: int64


---
## Step 5: Clean Text Fields

Strip whitespace from:
- title
- phrase
- article_text

In [165]:
# Clean text fields by stripping whitespace
df_clean['title'] = df_clean['title'].fillna('').astype(str).str.strip()
df_clean['phrase'] = df_clean['phrase'].fillna('').astype(str).str.strip()
df_clean['article_text'] = df_clean['article_text_clean']  # Use the cleaned version

# Drop the temporary cleaned column
df_clean = df_clean.drop(columns=['article_text_clean'])

print("Text fields cleaned!")
print(f"Sample title: {df_clean['title'].iloc[0][:80]}...")
print(f"Sample phrase: {df_clean['phrase'].iloc[0]}")

Text fields cleaned!
Sample title: Year in review: A look at news events in February 2024...
Sample phrase: elderly population


---
## Step 6: Clean and Validate phrase_category

Standardize phrase_category to one of:
- limiting
- neutral
- empowering

Any unrecognized values default to 'neutral'.

In [166]:
# Check current phrase_category values
print("Current phrase_category values:")
print(df_clean['phrase_category'].value_counts())

Current phrase_category values:
phrase_category
neutral       8415
empowering    3855
limiting      1668
Name: count, dtype: int64


In [167]:
def standardize_phrase_category(category):
    """Standardize phrase_category following Otto's logic"""
    if pd.isna(category):
        return 'neutral'
    
    category_clean = str(category).lower().strip()
    
    if category_clean == 'limiting':
        return 'limiting'
    elif category_clean == 'neutral':
        return 'neutral'
    elif category_clean == 'empowering':
        return 'empowering'
    else:
        return 'neutral'  # Default to neutral

df_clean['phrase_category_clean'] = df_clean['phrase_category'].apply(standardize_phrase_category)

print("\nStandardized phrase_category values:")
print(df_clean['phrase_category_clean'].value_counts())


Standardized phrase_category values:
phrase_category_clean
neutral       8415
empowering    3855
limiting      1668
Name: count, dtype: int64


---
## Step 7: Date Standardization

Parse publish_date and extract:
- year
- month

This helps with temporal analysis of the articles.

In [168]:
# Parse publish_date to datetime
df_clean['publish_date'] = pd.to_datetime(df_clean['publish_date'], errors='coerce')

# Extract year and month
df_clean['year'] = df_clean['publish_date'].dt.year
df_clean['month'] = df_clean['publish_date'].dt.month

print("Date standardization complete!")
print(f"\nDate range: {df_clean['publish_date'].min()} to {df_clean['publish_date'].max()}")
print(f"\nArticles by year:")
print(df_clean['year'].value_counts().sort_index())
print(f"\nArticles by month (2024):")
print(df_clean[df_clean['year'] == 2024]['month'].value_counts().sort_index())

Date standardization complete!

Date range: 2025-01-01 00:00:00 to 2026-01-30 00:00:00

Articles by year:
year
2025    11378
2026     2560
Name: count, dtype: int64

Articles by month (2024):
Series([], Name: count, dtype: int64)


---
## Step 8: Add Metadata

Add metadata columns:
- **ingestion_date**: Date when cleaning was performed
- **row_hash**: Unique hash based on URL for data tracking

In [169]:
# Add ingestion date
df_clean['ingestion_date'] = datetime.now().date()

# Create row hash from URL
def create_row_hash(url):
    """Create a hash from URL for unique identification"""
    if pd.isna(url):
        return 'hash_unknown'
    return 'hash_' + hashlib.md5(str(url).encode()).hexdigest()[:12]

df_clean['row_hash'] = df_clean['url'].apply(create_row_hash)

print("Metadata added!")
print(f"Ingestion date: {df_clean['ingestion_date'].iloc[0]}")
print(f"Sample row_hash: {df_clean['row_hash'].iloc[0]}")

Metadata added!
Ingestion date: 2026-02-02
Sample row_hash: hash_1a25b6f9e7b4


---
## Step 9: Final Cleanup and Summary

Reorder columns and display the cleaning summary.

In [170]:
# Reorder columns for clarity
column_order = [
    'url', 'title', 'publish_date', 'year', 'month',
    'media_name', 'phrase', 'phrase_category', 'phrase_category_clean',
    'province', 'province_clean', 'article_text',
    'ingestion_date', 'row_hash'
]

# Only include columns that exist
existing_columns = [col for col in column_order if col in df_clean.columns]
# Add any columns not in the order list
other_columns = [col for col in df_clean.columns if col not in existing_columns]
df_clean = df_clean[existing_columns + other_columns]

final_count = len(df_clean)

print("=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)
print(f"Initial rows:                    {initial_count}")
print(f"Removed (NULL URLs):             {initial_count - after_null_filter}")
print(f"Removed (Duplicates):            {after_null_filter - after_dedup}")
print(f"Removed (Invalid article text):  {after_dedup - after_validation}")
print(f"Final clean rows:                {final_count}")
print("-" * 60)
print(f"Total removed:                   {initial_count - final_count}")
print(f"Retention rate:                  {(final_count/initial_count*100):.1f}%")
print("=" * 60)

CLEANING SUMMARY
Initial rows:                    16149
Removed (NULL URLs):             0
Removed (Duplicates):            832
Removed (Invalid article text):  1379
Final clean rows:                13938
------------------------------------------------------------
Total removed:                   2211
Retention rate:                  86.3%


In [171]:
# Display final dataset structure
print("\nFinal Dataset Structure:")
print("-" * 60)
print(f"Rows: {len(df_clean)}")
print(f"Columns: {len(df_clean.columns)}")
print(f"\nColumn names: {list(df_clean.columns)}")
print("\nData Types:")
print(df_clean.dtypes)


Final Dataset Structure:
------------------------------------------------------------
Rows: 13938
Columns: 14

Column names: ['url', 'title', 'publish_date', 'year', 'month', 'media_name', 'phrase', 'phrase_category', 'phrase_category_clean', 'province', 'province_clean', 'article_text', 'ingestion_date', 'row_hash']

Data Types:
url                                 str
title                               str
publish_date             datetime64[us]
year                              int32
month                             int32
media_name                          str
phrase                              str
phrase_category                     str
phrase_category_clean               str
province                            str
province_clean                      str
article_text                        str
ingestion_date                   object
row_hash                            str
dtype: object


In [172]:
# Preview the cleaned data
df_clean.head()

,url,title,publish_date,year,month,media_name,phrase,phrase_category,phrase_category_clean,province,province_clean,article_text,ingestion_date,row_hash
13549,https://www.castanet.net/news/Canada/525368/Ye...,Year in review: A look at news events in Febru...,2025-01-01,2025,1,castanet.net,elderly population,neutral,neutral,British Columbia,British Columbia,Canada News \n\nYear in review: A look at new...,2026-02-02,hash_1a25b6f9e7b4
13551,https://www.nsnews.com/automotive/year-in-revi...,Year in review: A look at news events in Febru...,2025-01-01,2025,1,nsnews.com,elderly population,neutral,neutral,British Columbia,British Columbia,Skip to content\n\n×\n\nSupport Us\n\nSign in ...,2026-02-02,hash_5d293b43407a
13833,https://gulfislandsdriftwood.com/crd-lcc-cap-o...,CRD/LCC cap off busy year,2025-01-01,2025,1,gulfislandsdriftwood.com,active aging,empowering,empowering,British Columbia,British Columbia,Members of the first Salt Spring Island Local ...,2026-02-02,hash_e88bc4766804
4716,https://www.thestar.com/news/canada/year-in-re...,Year in review: A look at news events in Febru...,2025-01-01,2025,1,thestar.com,elderly population,neutral,neutral,Ontario,Ontario,Canada’s East Coast was hit by several snowsto...,2026-02-02,hash_8559340d28bd
4717,https://www.niagarafallsreview.ca/news/canada/...,Year in review: A look at news events in Febru...,2025-01-01,2025,1,niagarafallsreview.ca,elderly population,neutral,neutral,Ontario,Ontario,A look at news events in February 2024: 01 - T...,2026-02-02,hash_155886a469d1


---
## Step 10: Save Cleaned Data

Save the cleaned dataset to a new CSV file.

In [173]:
# Save the cleaned dataset
df_clean.to_csv(OUTPUT_FILE, index=False)

print(f"Cleaned data saved to: {OUTPUT_FILE}")
print(f"Total records: {len(df_clean)}")

Cleaned data saved to: ../newData/cleanData/aagingnarrativesarticles20252026_cleaned.csv
Total records: 13938


---
## Bonus: Distribution Analysis

Quick analysis of the cleaned data distributions.

In [174]:
print("Distribution by Province:")
print("-" * 40)
province_dist = df_clean['province_clean'].value_counts()
for province, count in province_dist.items():
    pct = count / len(df_clean) * 100
    print(f"{province}: {count} ({pct:.1f}%)")

Distribution by Province:
----------------------------------------
Ontario: 7041 (50.5%)
British Columbia: 4361 (31.3%)
Alberta: 2386 (17.1%)
Quebec: 150 (1.1%)


In [175]:
print("\nDistribution by Phrase Category:")
print("-" * 40)
category_dist = df_clean['phrase_category_clean'].value_counts()
for category, count in category_dist.items():
    pct = count / len(df_clean) * 100
    print(f"{category}: {count} ({pct:.1f}%)")


Distribution by Phrase Category:
----------------------------------------
neutral: 8415 (60.4%)
empowering: 3855 (27.7%)
limiting: 1668 (12.0%)


In [176]:
print("\nTop 10 Media Sources:")
print("-" * 40)
media_dist = df_clean['media_name'].value_counts().head(10)
for media, count in media_dist.items():
    print(f"{media}: {count}")


Top 10 Media Sources:
----------------------------------------
niagarafallsreview.ca: 1234
wellandtribune.ca: 1202
therecord.com: 1126
thespec.com: 930
citynews.ca: 588
thestar.com: 559
theglobeandmail.com: 456
cbc.ca: 404
rmoutlook.com: 393
townandcountrytoday.com: 390


In [177]:
print("\nArticle Text Length Statistics:")
print("-" * 40)
text_lengths = df_clean['article_text'].str.len()
print(f"Min length:    {text_lengths.min()} characters")
print(f"Max length:    {text_lengths.max()} characters")
print(f"Mean length:   {text_lengths.mean():.0f} characters")
print(f"Median length: {text_lengths.median():.0f} characters")


Article Text Length Statistics:
----------------------------------------
Min length:    216 characters
Max length:    5000 characters
Mean length:   4253 characters
Median length: 4904 characters


---
## Done!

The 2024 articles dataset has been cleaned using Otto's techniques. The cleaned data is saved to:
`newData/aging_narratives_articles_2024_cleaned.csv`